# Threat hunting playbook

A minimal **watchlist + join + timeline** hunt on bundled VPC flow telemetry. The steps below walk through a common analyst pattern: discover candidates, freeze a watchlist, then ask *when* those IPs show up in traffic.

## What this notebook simulates

This is a **teaching example**, not a live incident. It simulates a short threat-hunting workflow an analyst might run after noticing unusual outbound or inbound VPC traffic:

1. **Discover** — scan flow logs for the busiest **external** (non–RFC 1918) destination IPs.
2. **Decide** — pick six IPs worth watching and **publish** them as a reusable watchlist.
3. **Investigate** — pull only flows involving those IPs and **bucket them over time** to see activity spikes.

All of that runs on Cribl’s bundled sample data so you do not need external feeds or dataset providers.

### Main dataset

**`cribl_search_sample`** is the primary Search dataset. Within it, rows with `dataSource == "vpcflowlogs"` stand in for **Amazon VPC flow log** records: who talked to whom (`srcaddr`, `dstaddr`), when (`_time`), and volume. That slice is the **telemetry** you hunt over—the “ground truth” event stream.

### Role of the lookup

**`notebook_vpc_external_watchlist.csv`** is a **static Search lookup** (saved via `%%cribl_save_search_lookup`). It is not more telemetry; it is a small **reference table** you control:

| Column | Meaning |
|--------|---------|
| `ip_address` | The six external destinations you chose in §A |
| `flow_count` | How often each IP appeared when you built the list (context only) |
| `watchlist` | Label (`vpc_external_dst`) for documentation |

Lookups act like a **portable IOC / watchlist**: cheap to store, easy to share, and queryable from KQL via `$vt_lookups`. In production you might load the same file from a ticket, a TI feed, or an export—here we build it from the sample flows themselves.

### What the join query represents

§C runs an **inner join** between:

- **Left:** VPC flow events from `cribl_search_sample` (`dstaddr` = destination IP on each flow).
- **Right:** Your lookup (`ip_address` = the watchlisted IPs).

**Only flows whose destination is on the watchlist are kept.** That models the hunt question: *“Show me all traffic **to** these suspicious external IPs—not every flow in the account.”*

`timestats span=1h count() by ip_address` then rolls those matching flows into **hourly counts per IP**, so you can see **when** each watchlisted address was active in the sample window (the timeline chart in §D).

**Artifacts:** lookup file `notebook_vpc_external_watchlist.csv` (removed in **Cleanup**).

## Prerequisites

1. Hosted Cribl app with **Cribl Search** and lookup magics (`%%cribl_save_search_lookup`, etc.).
2. No external URLs or dataset providers — everything uses **`cribl_search_sample`**.

**Related:** `Malware_Hash_Threat_Hunt.ipynb` (MD5 + TI lookup + HTTP dataset provider), `Cribl_Search_Lookup_Magics.ipynb`, `Cribl_Search_Examples.ipynb`, `Incident_Triage_Playbook.ipynb`.

### A) Select six external destination IPs

From VPC flows, rank **`dstaddr`** values that are valid IPv4 and **not** in private ranges (`ipv4_is_private`).

In [ ]:
%%cribl_search var=external_ip_candidates lang=kql limit=50 preview=true earliest=-1h latest=now timeout=180
dataset=cribl_search_sample
| where dataSource == "vpcflowlogs"
| where isnotnull(parse_ipv4(tostring(dstaddr))) and not(ipv4_is_private(tostring(dstaddr)))
| summarize flow_count = count() by ip_address = tostring(dstaddr)
| top 6 by flow_count desc

### B) Build and publish the static lookup

In [ ]:
import pandas as pd

def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None

ip_col = _pick_col(external_ip_candidates, "ip_address", "dstaddr")
count_col = _pick_col(external_ip_candidates, "flow_count", "count")
if ip_col is None:
    raise ValueError(
        f"Expected ip_address from Search; got {list(external_ip_candidates.columns)}. Re-run §A."
    )
if len(external_ip_candidates) == 0:
    raise ValueError("No external IPs returned — check dataSource filter or Search availability.")

watchlist_df = external_ip_candidates[[ip_col] + ([count_col] if count_col else [])].copy()
watchlist_df = watchlist_df.rename(columns={ip_col: "ip_address"})
if count_col:
    watchlist_df = watchlist_df.rename(columns={count_col: "flow_count"})
watchlist_df["watchlist"] = "vpc_external_dst"
watchlist_df = watchlist_df.drop_duplicates(subset=["ip_address"]).head(6)

print("watchlist rows:", len(watchlist_df))
watchlist_df

In [ ]:
%%cribl_save_search_lookup notebook_vpc_external_watchlist.csv var=watchlist_df replace=true

In [ ]:
%%cribl_search var=lookup_preview lang=kql limit=10 preview=true earliest=-1h latest=now
dataset="$vt_lookups" lookupFile="notebook_vpc_external_watchlist"
| limit 10

### C) Join VPC flows to the watchlist + `timestats`

Re-reads **`cribl_search_sample`** (main telemetry), **inner-joins** to `$vt_lookups` so only flows with `dstaddr` on the watchlist remain, then **`timestats span=1h count() by ip_address`** (put `span=` before the aggregation). Do not start with `let` — `%%cribl_search` prepends `cribl` only to queries that begin with `dataset=`.

In [ ]:
%%cribl_search var=timeline lang=kql limit=5000 preview=true earliest=-1h latest=now timeout=300
dataset=cribl_search_sample
| where dataSource == "vpcflowlogs"
| join kind=inner (
    dataset="$vt_lookups" lookupFile="notebook_vpc_external_watchlist"
  ) on $left.dstaddr == $right.ip_address
| timestats span=1h count() by ip_address

### D) Visualize timeline

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None

if len(timeline) == 0:
    raise ValueError("timeline is empty — re-run §A–C (lookup must exist before join).")

time_col = _pick_col(timeline, "_time", "time", "timestamp")
ip_col = _pick_col(timeline, "ip_address", "dstaddr")
count_col = _pick_col(timeline, "count_", "count", "flows")

if time_col is None:
    raise ValueError(f"Expected a time column; got {list(timeline.columns)}")

if ip_col is not None:
    plot_df = timeline[[time_col, ip_col] + ([count_col] if count_col else [])].copy()
    if not count_col:
        count_col = "_n"
        plot_df[count_col] = 1
else:
    # Cribl timestats often returns wide rows: _time + one column per group-by value (IP).
    value_cols = [c for c in timeline.columns if c != time_col]
    if not value_cols:
        raise ValueError(f"Expected ip_address or wide IP columns; got {list(timeline.columns)}")
    plot_df = timeline.melt(
        id_vars=[time_col], value_vars=value_cols, var_name="ip_address", value_name="count"
    )
    ip_col = "ip_address"
    count_col = "count"

plot_df[time_col] = pd.to_datetime(plot_df[time_col], errors="coerce")
plot_df = plot_df.dropna(subset=[time_col])
plot_df[count_col] = pd.to_numeric(plot_df[count_col], errors="coerce").fillna(0)

print("timestats buckets:", len(plot_df), "| IPs:", plot_df[ip_col].nunique())

fig, ax = plt.subplots(figsize=(10, 4))
for ip, grp in plot_df.groupby(ip_col):
    g = grp.sort_values(time_col)
    ax.plot(g[time_col], g[count_col], marker="o", linewidth=1.5, label=str(ip))
ax.set_title("Watchlisted external IPs — flow hits over time (timestats 1h)")
ax.set_xlabel("Time")
ax.set_ylabel("Flow count")
ax.legend(loc="upper right", fontsize=8, ncol=2)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(
    "Verdict: joined VPC flows show when each watchlisted external IP appears in the sample window."
)
plot_df.head(12)

### Cleanup

In [ ]:
%%cribl_delete_search_lookup notebook_vpc_external_watchlist.csv

### Next steps

- Swap `dataSource == "vpcflowlogs"` for your production VPC dataset name
- Add `srcaddr` join (union) if you also want hits when watchlisted IPs are sources
- `Cribl_Search_Lookup_Magics.ipynb` — lookup lifecycle
- `Cribl_Search_Examples.ipynb` — more KQL patterns